# LowLightGAN — LOL Dataset Training
**Add Data:** soumikrakshit/lol-dataset
**Accelerator:** GPU T4 x2

In [ ]:
# Cell 1 — Config
CONFIG = {
    # LOL Dataset paths on Kaggle
    'lol_train_low'  : '/kaggle/input/lol-dataset/our485/low',
    'lol_train_high' : '/kaggle/input/lol-dataset/our485/high',
    'lol_test_low'   : '/kaggle/input/lol-dataset/eval15/low',
    'lol_test_high'  : '/kaggle/input/lol-dataset/eval15/high',
    'save_dir'       : '/kaggle/working/checkpoints',
    'sample_dir'     : '/kaggle/working/samples',
    'pretrain_epochs': 30,
    'joint_epochs'   : 200,
    'batch_size'     : 8,
    'patch_size'     : 256,
    'lr'             : 2e-4,
    'alpha'          : 0.1,
    'beta'           : 0.01,
    'gamma'          : 0.5,
    'delta'          : 0.1,
    'num_workers'    : 2,
    'save_every'     : 10,
    'sample_every'   : 5,
    'device'         : 'cuda',
}
print('Config OK')

Config OK


In [ ]:
# Cell 2 — Imports
import os, time, random
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR
import torchvision.transforms.functional as TF
import torchvision.transforms as T
import torchvision.models as tv_models
print('Imports OK')

Imports OK


In [ ]:
# Cell 3 — LOL Dataset (Paired)
class LOLDataset(Dataset):
    def __init__(self, low_dir, high_dir, patch_size=256, augment=True):
        self.patch   = patch_size
        self.augment = augment
        exts = {'.png','.jpg','.jpeg','.bmp'}
        self.low_paths  = sorted([p for p in Path(low_dir).iterdir()  if p.suffix.lower() in exts])
        self.high_paths = sorted([p for p in Path(high_dir).iterdir() if p.suffix.lower() in exts])
        assert len(self.low_paths)==len(self.high_paths) and len(self.low_paths)>0
        print(f'[LOL] {len(self.low_paths)} pairs  augment={augment}')

    def __len__(self): return len(self.low_paths)

    def _crop(self, l, h):
        _, H, W = l.shape
        if H<self.patch or W<self.patch:
            l = TF.resize(l,[self.patch,self.patch],antialias=True)
            h = TF.resize(h,[self.patch,self.patch],antialias=True)
            return l, h
        top  = random.randint(0, H-self.patch)
        left = random.randint(0, W-self.patch)
        return TF.crop(l,top,left,self.patch,self.patch), TF.crop(h,top,left,self.patch,self.patch)

    def _augment(self, l, h):
        if random.random()>0.5: l=TF.hflip(l); h=TF.hflip(h)
        if random.random()>0.5: l=TF.vflip(l); h=TF.vflip(h)
        angle=random.uniform(-10,10)
        l=TF.rotate(l,angle); h=TF.rotate(h,angle)
        l=T.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.1,hue=0.05)(l)
        return l, h

    def __getitem__(self, idx):
        l = TF.to_tensor(Image.open(self.low_paths[idx]).convert('RGB'))
        h = TF.to_tensor(Image.open(self.high_paths[idx]).convert('RGB'))
        l, h = self._crop(l, h)
        if self.augment: l, h = self._augment(l, h)
        return {'low':l.clamp(0,1), 'high':h.clamp(0,1), 'name':self.low_paths[idx].stem}

print('Dataset class OK')

Dataset class OK


In [ ]:
# Cell 4 — Dataset
class DarkFaceDataset(Dataset):
    def __init__(self, images_dir, labels_dir, img_size=256, augment=True):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.img_size   = img_size
        self.augment    = augment
        if not self.images_dir.exists():
            raise FileNotFoundError(f'Not found: {self.images_dir}')
        exts = {'.jpg', '.jpeg', '.png', '.bmp'}
        self.paths = sorted([p for p in self.images_dir.iterdir() if p.suffix.lower() in exts])
        assert len(self.paths) > 0
        print(f'[DarkFace] {len(self.paths)} images  img_size={img_size}')

    def __len__(self): return len(self.paths)

    def _load_boxes(self, img_path, sx, sy):
        lp = self.labels_dir / (img_path.stem + '.txt')
        boxes = []
        if lp.exists():
            lines = lp.read_text().strip().splitlines()
            if lines:
                n = int(lines[0])
                for line in lines[1:n+1]:
                    p = list(map(int, line.split()))
                    if len(p) == 4:
                        boxes.append([int(p[0]*sx), int(p[1]*sy), int(p[2]*sx), int(p[3]*sy)])
        return boxes

    def _augment(self, img_t):
        if random.random() > 0.5: img_t = TF.hflip(img_t)
        if random.random() > 0.5: img_t = TF.vflip(img_t)
        img_t = TF.rotate(img_t, random.uniform(-10, 10))
        img_t = T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05)(img_t)
        return img_t

    def __getitem__(self, idx):
        img    = Image.open(self.paths[idx]).convert('RGB')
        ow, oh = img.size
        img    = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        boxes  = self._load_boxes(self.paths[idx], self.img_size/ow, self.img_size/oh)
        img_t  = TF.to_tensor(img)
        if self.augment: img_t = self._augment(img_t)
        return {'image': img_t.clamp(0,1), 'boxes': boxes, 'num_faces': len(boxes), 'name': self.paths[idx].stem}

def collate_fn(batch):
    return {'image': torch.stack([b['image'] for b in batch]),
            'boxes': [b['boxes'] for b in batch],
            'num_faces': [b['num_faces'] for b in batch],
            'name': [b['name'] for b in batch]}
print('Dataset OK')

Dataset OK


In [ ]:
# Cell 5 — Model
class IlluminationCurve(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1,16,3,padding=1), nn.LeakyReLU(0.2,True),
                                 nn.Conv2d(16,16,3,padding=1), nn.LeakyReLU(0.2,True), nn.Conv2d(16,1,1))
    def forward(self, L): return torch.sigmoid(L + self.net(L))

class RetinexBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Conv2d(3,32,3,padding=1), nn.LeakyReLU(0.2,True),
                                     nn.Conv2d(32,64,3,padding=1), nn.LeakyReLU(0.2,True),
                                     nn.Conv2d(64,64,3,padding=1), nn.LeakyReLU(0.2,True))
        self.illum_head   = nn.Sequential(nn.Conv2d(64,32,3,padding=1), nn.LeakyReLU(0.2,True), nn.Conv2d(32,1,1), nn.Sigmoid())
        self.reflect_head = nn.Sequential(nn.Conv2d(64,32,3,padding=1), nn.LeakyReLU(0.2,True), nn.Conv2d(32,3,1), nn.Sigmoid())
        self.illum_enhance = IlluminationCurve()
    def forward(self, x):
        f = self.encoder(x); L = self.illum_head(f); R = self.reflect_head(f)
        return R, L, self.illum_enhance(L)

class DoubleConv(nn.Module):
    def __init__(self,i,o):
        super().__init__()
        self.b = nn.Sequential(nn.Conv2d(i,o,3,padding=1,bias=False), nn.BatchNorm2d(o), nn.LeakyReLU(0.2,True),
                               nn.Conv2d(o,o,3,padding=1,bias=False), nn.BatchNorm2d(o), nn.LeakyReLU(0.2,True))
    def forward(self,x): return self.b(x)

class Down(nn.Module):
    def __init__(self,i,o):
        super().__init__(); self.b = nn.Sequential(nn.MaxPool2d(2), DoubleConv(i,o))
    def forward(self,x): return self.b(x)

class Up(nn.Module):
    def __init__(self,i,o):
        super().__init__(); self.up = nn.ConvTranspose2d(i,i//2,2,stride=2); self.conv = DoubleConv(i,o)
    def forward(self,x,skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]: x = F.interpolate(x,size=skip.shape[2:],mode='bilinear',align_corners=False)
        return self.conv(torch.cat([skip,x],1))

class NoiseEstimator(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.inc=DoubleConv(3,base); self.d1=Down(base,base*2); self.d2=Down(base*2,base*4); self.d3=Down(base*4,base*8)
        self.bot=DoubleConv(base*8,base*16)
        self.u1=Up(base*16,base*8); self.u2=Up(base*8,base*4); self.u3=Up(base*4,base*2); self.u4=Up(base*2,base)
        self.out=nn.Sequential(nn.Conv2d(base,1,1),nn.Sigmoid())
    def forward(self,x):
        x1=self.inc(x); x2=self.d1(x1); x3=self.d2(x2); x4=self.d3(x3)
        x=self.bot(x4); x=self.u1(x,x4); x=self.u2(x,x3); x=self.u3(x,x2); x=self.u4(x,x1)
        return self.out(x)

class NAABlock(nn.Module):
    def __init__(self,channels,r=8):
        super().__init__()
        self.T=nn.Parameter(torch.tensor(0.1)); red=max(channels//r,1)
        self.mlp=nn.Sequential(nn.Linear(channels,red,bias=False),nn.ReLU(True),nn.Linear(red,channels,bias=False),nn.Sigmoid())
        self.sp=nn.Sequential(nn.Conv2d(2,1,7,padding=3,bias=False),nn.Sigmoid())
    def forward(self,F_feat,sigma):
        if sigma.shape[2:]!=F_feat.shape[2:]: sigma=F.interpolate(sigma,size=F_feat.shape[2:],mode='bilinear',align_corners=False)
        M=torch.sigmoid(-sigma/self.T.clamp(1e-4)); Fm=F_feat*M
        A_ch=self.mlp(Fm.mean([2,3])).unsqueeze(-1).unsqueeze(-1)
        A_sp=self.sp(torch.cat([Fm.mean(1,True),Fm.max(1,True)[0]],1))
        return F_feat+F_feat*A_ch*A_sp

class ConvBNReLU(nn.Module):
    def __init__(self,i,o):
        super().__init__()
        self.b=nn.Sequential(nn.Conv2d(i,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.LeakyReLU(0.2,True))
    def forward(self,x): return self.b(x)

class ResBlock(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.b=nn.Sequential(nn.Conv2d(c,c,3,padding=1,bias=False),nn.BatchNorm2d(c),nn.ReLU(True),
                             nn.Conv2d(c,c,3,padding=1,bias=False),nn.BatchNorm2d(c))
    def forward(self,x): return F.relu(x+self.b(x))

class EncBlock(nn.Module):
    def __init__(self,i,o):
        super().__init__(); self.conv=nn.Sequential(ConvBNReLU(i,o),ConvBNReLU(o,o)); self.pool=nn.MaxPool2d(2)
    def forward(self,x): s=self.conv(x); return self.pool(s),s

class DecBlock(nn.Module):
    def __init__(self,i,skip_ch,o):
        super().__init__()
        self.up=nn.ConvTranspose2d(i,i//2,2,stride=2)
        self.conv=nn.Sequential(ConvBNReLU(i//2+skip_ch,o),ConvBNReLU(o,o))
        self.naa=NAABlock(o)
    def forward(self,x,skip,sigma):
        x=self.up(x)
        if x.shape[2:]!=skip.shape[2:]: x=F.interpolate(x,size=skip.shape[2:],mode='bilinear',align_corners=False)
        return self.naa(self.conv(torch.cat([skip,x],1)),sigma)

class Generator(nn.Module):
    def __init__(self,base=64,n_res=3):
        super().__init__()
        self.e1=EncBlock(4,base); self.e2=EncBlock(base,base*2)
        self.e3=EncBlock(base*2,base*4); self.e4=EncBlock(base*4,base*8)
        self.bot=nn.Sequential(ConvBNReLU(base*8,base*8),*[ResBlock(base*8) for _ in range(n_res)],ConvBNReLU(base*8,base*8))
        self.d4=DecBlock(base*8,base*8,base*4); self.d3=DecBlock(base*4,base*4,base*2)
        self.d2=DecBlock(base*2,base*2,base);   self.d1=DecBlock(base,base,base//2)
        self.out=nn.Sequential(nn.Conv2d(base//2,3,1),nn.Sigmoid())
    def forward(self,R,sigma):
        x=torch.cat([R,sigma],1); x,s1=self.e1(x); x,s2=self.e2(x); x,s3=self.e3(x); x,s4=self.e4(x)
        x=self.bot(x); x=self.d4(x,s4,sigma); x=self.d3(x,s3,sigma); x=self.d2(x,s2,sigma); x=self.d1(x,s1,sigma)
        return self.out(x)

class Discriminator(nn.Module):
    def __init__(self,base=64):
        super().__init__()
        def cb(i,o,bn=True):
            l=[nn.Conv2d(i,o,4,stride=2,padding=1,bias=not bn)]
            if bn: l.append(nn.BatchNorm2d(o))
            l.append(nn.LeakyReLU(0.2,True)); return l
        self.model=nn.Sequential(*cb(3,base,False),*cb(base,base*2),*cb(base*2,base*4),
                                 nn.Conv2d(base*4,base*8,4,stride=1,padding=1,bias=False),
                                 nn.BatchNorm2d(base*8),nn.LeakyReLU(0.2,True),
                                 nn.Conv2d(base*8,1,4,stride=1,padding=1))
    def forward(self,x): return self.model(x)

class LowLightGAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.retinex=RetinexBranch(); self.noise_est=NoiseEstimator(); self.generator=Generator()
    def forward(self,low):
        R,L,L_enh=self.retinex(low); sigma=self.noise_est(low); enhanced=self.generator(R,sigma)
        return {'R':R,'L':L,'L_enhanced':L_enh,'sigma':sigma,'enhanced':enhanced,'retinex_out':R*L_enh}

print('Model OK')

Model OK


In [ ]:
# Cell 6 — Setup (Dataset + Model)
import os
for root, dirs, files in os.walk('/kaggle/input'):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/soumikrakshit
/kaggle/input/datasets/soumikrakshit/lol-dataset
/kaggle/input/datasets/soumikrakshit/lol-dataset/lol_dataset
/kaggle/input/datasets/soumikrakshit/lol-dataset/lol_dataset/eval15
/kaggle/input/datasets/soumikrakshit/lol-dataset/lol_dataset/eval15/low
/kaggle/input/datasets/soumikrakshit/lol-dataset/lol_dataset/eval15/high
/kaggle/input/datasets/soumikrakshit/lol-dataset/lol_dataset/our485
/kaggle/input/datasets/soumikrakshit/lol-dataset/lol_dataset/our485/low
/kaggle/input/datasets/soumikrakshit/lol-dataset/lol_dataset/our485/high


In [ ]:
#cell 7
import torch
from pathlib import Path
from torch.utils.data import DataLoader

cfg    = CONFIG
cfg['lol_train_low']  = '/kaggle/input/datasets/soumikrakshit/lol-dataset/lol_dataset/our485/low'
cfg['lol_train_high'] = '/kaggle/input/datasets/soumikrakshit/lol-dataset/lol_dataset/our485/high'
cfg['lol_test_low']   = '/kaggle/input/datasets/soumikrakshit/lol-dataset/lol_dataset/eval15/low'
cfg['lol_test_high']  = '/kaggle/input/datasets/soumikrakshit/lol-dataset/lol_dataset/eval15/high'
device = torch.device(cfg['device'] if torch.cuda.is_available() else 'cpu')

Path(cfg['save_dir']).mkdir(parents=True, exist_ok=True)
Path(cfg['sample_dir']).mkdir(parents=True, exist_ok=True)

train_dataset = LOLDataset(cfg['lol_train_low'], cfg['lol_train_high'],
                            patch_size=cfg['patch_size'], augment=True)
val_dataset   = LOLDataset(cfg['lol_test_low'], cfg['lol_test_high'],
                            patch_size=cfg['patch_size'], augment=False)

train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'],
                          shuffle=True, num_workers=cfg['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=1,
                          shuffle=False, num_workers=cfg['num_workers'], pin_memory=True)

G = LowLightGAN().to(device)
D = Discriminator().to(device)
print(f'Device: {device}')
print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} batches')
print('Ready ✓')

[LOL] 485 pairs  augment=True
[LOL] 15 pairs  augment=False
Device: cuda
Train: 61 batches | Val: 15 batches
Ready ✓


In [ ]:
# Cell 8 — Pre-Training
import time, torch.optim as optim, torch.nn.functional as F
from skimage.metrics import structural_similarity as sk_ssim

def compute_psnr(pred, target):
    mse = F.mse_loss(pred, target, reduction='none').mean(dim=[1,2,3])
    return (10*torch.log10(1.0/mse.clamp(1e-10))).mean().item()

def compute_ssim(pred, target):
    p = pred.detach().cpu().permute(0,2,3,1).numpy()
    t = target.detach().cpu().permute(0,2,3,1).numpy()
    return float(np.mean([sk_ssim(p[i],t[i],data_range=1.0,channel_axis=2) for i in range(len(p))]))

print(f'\n{"="*55}\n  PRE-TRAINING ({cfg["pretrain_epochs"]} epochs)\n{"="*55}')
params = list(G.retinex.parameters()) + list(G.noise_est.parameters())
opt    = optim.Adam(params, lr=cfg['lr'], betas=(0.9,0.999))

for ep in range(1, cfg['pretrain_epochs']+1):
    G.train(); total=0.0; t0=time.time()
    for bidx, batch in enumerate(train_loader):
        low  = batch['low'].to(device)
        high = batch['high'].to(device)
        opt.zero_grad()
        out  = G(low)
        # Paired: retinex_out should match high
        l_ret  = F.l1_loss(out['retinex_out'], high)
        l_cons = F.l1_loss(out['R']*out['L'], low)
        loss   = l_ret + 0.5*l_cons
        loss.backward(); opt.step(); total+=loss.item()
        if (bidx+1)%50==0: print(f'  [{bidx+1}/{len(train_loader)}] loss={loss.item():.4f}')
    print(f'Epoch {ep:3d}/{cfg["pretrain_epochs"]}  loss={total/len(train_loader):.4f}  {time.time()-t0:.1f}s')
    torch.save({'ep':ep,'retinex':G.retinex.state_dict(),'noise':G.noise_est.state_dict()},
               f'{cfg["save_dir"]}/pretrain_ep{ep:03d}.pth')

print('Pre-training done ✓')


  PRE-TRAINING (30 epochs)
  [50/61] loss=0.2176
Epoch   1/30  loss=0.2419  18.6s
  [50/61] loss=0.2026
Epoch   2/30  loss=0.1774  17.1s
  [50/61] loss=0.1630
Epoch   3/30  loss=0.1528  17.6s
  [50/61] loss=0.1440
Epoch   4/30  loss=0.1482  18.3s
  [50/61] loss=0.1262
Epoch   5/30  loss=0.1469  19.0s
  [50/61] loss=0.1502
Epoch   6/30  loss=0.1508  20.4s
  [50/61] loss=0.1892
Epoch   7/30  loss=0.1455  21.7s
  [50/61] loss=0.1824
Epoch   8/30  loss=0.1492  20.7s
  [50/61] loss=0.1096
Epoch   9/30  loss=0.1474  20.1s
  [50/61] loss=0.1772
Epoch  10/30  loss=0.1486  20.3s
  [50/61] loss=0.1401
Epoch  11/30  loss=0.1472  21.0s
  [50/61] loss=0.1338
Epoch  12/30  loss=0.1472  20.7s
  [50/61] loss=0.1339
Epoch  13/30  loss=0.1460  20.4s
  [50/61] loss=0.1249
Epoch  14/30  loss=0.1466  20.5s
  [50/61] loss=0.1186
Epoch  15/30  loss=0.1438  20.7s
  [50/61] loss=0.1532
Epoch  16/30  loss=0.1465  20.8s
  [50/61] loss=0.1488
Epoch  17/30  loss=0.1442  20.7s
  [50/61] loss=0.1358
Epoch  18/30  l

In [ ]:
# Cell 9 — Joint Training
import time, torch, torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import numpy as np
from PIL import Image
from torch.optim.lr_scheduler import LambdaLR
from pathlib import Path

class VGGLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg=tv_models.vgg16(weights=tv_models.VGG16_Weights.IMAGENET1K_V1)
        self.f=nn.Sequential(*list(vgg.features.children())[:9]).eval()
        for p in self.f.parameters(): p.requires_grad=False
        self.register_buffer('mean',torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer('std', torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
    def forward(self,p,t):
        p=self.f((p-self.mean)/self.std); t=self.f((t-self.mean)/self.std)
        return F.mse_loss(p,t)

def d_loss(D,real,fake): return 0.5*((D(real)-1)**2).mean()+0.5*(D(fake.detach())**2).mean()
def g_adv_loss(D,fake):  return 0.5*((D(fake)-1)**2).mean()

def save_ckpt(path,G,D,oG,oD,epoch,psnr=None):
    torch.save({'epoch':epoch,'G':G.state_dict(),'D':D.state_dict(),
                'oG':oG.state_dict(),'oD':oD.state_dict(),'psnr':psnr},path)
    print(f'  [Saved] {Path(path).name}  epoch={epoch}' + (f'  PSNR={psnr:.2f}' if psnr else ''))

def load_ckpt(path,G,D,oG,oD,device):
    ck=torch.load(path,map_location=device)
    G.load_state_dict(ck['G']); D.load_state_dict(ck['D'])
    oG.load_state_dict(ck['oG']); oD.load_state_dict(ck['oD'])
    print(f'  [Resumed] from epoch {ck["epoch"]}'); return ck['epoch']

def make_lr_lambda(total,start):
    def fn(ep): return 1.0 if ep<start else max(0.0,1.0-(ep-start)/max(1,total-start))
    return fn

@torch.no_grad()
def validate(G, loader, device):
    G.eval(); psnr_total=0.0; ssim_total=0.0; n=0
    for batch in loader:
        low  = batch['low'].to(device)
        high = batch['high'].to(device)
        enh  = G(low)['enhanced'].clamp(0,1)
        psnr_total += compute_psnr(enh, high)
        ssim_total += compute_ssim(enh, high)
        n += 1
    G.train()
    return psnr_total/n, ssim_total/n

@torch.no_grad()
def save_samples(G, loader, device, out_dir, epoch, n=8):
    G.eval(); Path(out_dir).mkdir(parents=True,exist_ok=True); saved=0
    for batch in loader:
        low  = batch['low'].to(device)
        high = batch['high'].to(device)
        enh  = G(low)['enhanced'].clamp(0,1)
        for i in range(len(batch['name'])):
            if saved>=n: break
            orig = Image.fromarray((low[i].cpu().permute(1,2,0).numpy()*255).astype(np.uint8))
            enhi = Image.fromarray((enh[i].cpu().permute(1,2,0).numpy()*255).astype(np.uint8))
            gt   = Image.fromarray((high[i].cpu().permute(1,2,0).numpy()*255).astype(np.uint8))
            comp = Image.new('RGB',(orig.width*3,orig.height))
            comp.paste(orig,(0,0)); comp.paste(enhi,(orig.width,0)); comp.paste(gt,(orig.width*2,0))
            comp.save(f'{out_dir}/ep{epoch:04d}_{batch["name"][i]}.jpg'); saved+=1
        if saved>=n: break
    print(f'  [Samples] {saved} saved  (low | enhanced | GT)'); G.train()

# ── Training ───────────────────────────────────────────────
print(f'\n{"="*55}\n  JOINT TRAINING ({cfg["joint_epochs"]} epochs)\n{"="*55}')
vgg = VGGLoss().to(device)
oG  = optim.Adam(G.parameters(), lr=cfg['lr'], betas=(0.5,0.999))
oD  = optim.Adam(D.parameters(), lr=cfg['lr'], betas=(0.9,0.999))
ep_total=cfg['joint_epochs']; decay=ep_total//2
sG=LambdaLR(oG,make_lr_lambda(ep_total,decay))
sD=LambdaLR(oD,make_lr_lambda(ep_total,decay))

resume_path=f'{cfg["save_dir"]}/latest_joint.pth'
start_ep=0; best_psnr=0.0
if Path(resume_path).exists(): start_ep=load_ckpt(resume_path,G,D,oG,oD,device)

for ep in range(start_ep+1, ep_total+1):
    G.train(); D.train(); lG_t=lD_t=0.0; t0=time.time()
    for bidx, batch in enumerate(train_loader):
        low  = batch['low'].to(device)
        high = batch['high'].to(device)
        # Discriminator
        oD.zero_grad()
        out=G(low); enh=out['enhanced']
        lD=d_loss(D,high,enh); lD.backward(); oD.step()
        # Generator
        oG.zero_grad()
        out=G(low); enh=out['enhanced']
        lL1    = F.l1_loss(enh, high)
        lAdv   = g_adv_loss(D, enh)
        lPerc  = vgg(enh, high)
        lNoise = F.relu(G.noise_est(enh)-G.noise_est(high)).mean()
        lG = lL1 + cfg['beta']*lAdv + cfg['alpha']*lPerc + cfg['gamma']*lNoise
        lG.backward(); oG.step()
        lG_t+=lG.item(); lD_t+=lD.item()
        if (bidx+1)%50==0: print(f'  [{bidx+1}/{len(train_loader)}] G={lG.item():.4f} D={lD.item():.4f}')
    sG.step(); sD.step(); N=len(train_loader)
    # Validate
    psnr, ssim = validate(G, val_loader, device)
    print(f'Epoch {ep:3d}/{ep_total}  G={lG_t/N:.4f}  D={lD_t/N:.4f}  PSNR={psnr:.2f}  SSIM={ssim:.4f}  {time.time()-t0:.1f}s')
    if ep%cfg['sample_every']==0: save_samples(G,train_loader,device,cfg['sample_dir'],ep)
    save_ckpt(resume_path,G,D,oG,oD,ep,psnr)
    if psnr>best_psnr:
        best_psnr=psnr
        save_ckpt(f'{cfg["save_dir"]}/best_model.pth',G,D,oG,oD,ep,psnr)
    if ep%cfg['save_every']==0: save_ckpt(f'{cfg["save_dir"]}/epoch_{ep:04d}.pth',G,D,oG,oD,ep,psnr)

save_ckpt(f'{cfg["save_dir"]}/final.pth',G,D,oG,oD,ep_total,best_psnr)
print(f'Training complete ✓  Best PSNR={best_psnr:.2f} dB')


  JOINT TRAINING (200 epochs)
  [Resumed] from epoch 4
  [50/61] G=0.2399 D=0.2863
Epoch   5/200  G=0.3253  D=0.0768  PSNR=17.95  SSIM=0.7442  106.2s
  [Samples] 8 saved  (low | enhanced | GT)
  [Saved] latest_joint.pth  epoch=5  PSNR=17.95
  [Saved] best_model.pth  epoch=5  PSNR=17.95
  [50/61] G=0.4177 D=0.0403
Epoch   6/200  G=0.3127  D=0.0742  PSNR=18.63  SSIM=0.7414  106.0s
  [Saved] latest_joint.pth  epoch=6  PSNR=18.63
  [Saved] best_model.pth  epoch=6  PSNR=18.63
  [50/61] G=0.2916 D=0.0777
Epoch   7/200  G=0.3050  D=0.1167  PSNR=16.68  SSIM=0.7303  105.8s
  [Saved] latest_joint.pth  epoch=7  PSNR=16.68
  [50/61] G=0.3358 D=0.0326
Epoch   8/200  G=0.2963  D=0.0804  PSNR=17.76  SSIM=0.7223  106.4s
  [Saved] latest_joint.pth  epoch=8  PSNR=17.76
  [50/61] G=0.3892 D=0.0759
Epoch   9/200  G=0.2924  D=0.0947  PSNR=18.25  SSIM=0.7575  106.2s
  [Saved] latest_joint.pth  epoch=9  PSNR=18.25
  [50/61] G=0.3626 D=0.0144
Epoch  10/200  G=0.2818  D=0.0934  PSNR=16.87  SSIM=0.7233  106.2s